# 🚀 Push YouTube-ASL (YTASL) T5 Checkpoints to Hugging Face

This notebook packages and pushes the **YouTube-ASL Sign Language Translation (SLT)** models from your Google Drive into standard, fully usable **Hugging Face Model Repositories**.

### 📦 Source Models in Google Drive (`YTASL - Checkpoint/`)
1. **`mt5-base`**: Multilingual T5 base for sign-language translation
2. **`t5-v1_1-base`**: T5 v1.1 base adapted for continuous pose features
3. **`t5-base`**: Standard T5 base adapted for pose-to-text generation

### 🛠️ What this notebook does:
* **Locates & inspects** the checkpoint directories in Google Drive (`YTASL - Checkpoint`).
* **Embeds Hugging Face compatibility code** (`modeling_t5.py`) so anyone can load models directly with:
  ```python
  from transformers import AutoModel
  model = AutoModel.from_pretrained("YourName/ytasl-t5-base", trust_remote_code=True)
  ```
* **Recursively discovers weights** (`.bin`, `.safetensors`, `.pt`) across subdirectories (e.g. `checkpoint-*`).
* **Converts weights to standard `model.safetensors`** with memory pointer deduplication for tied embeddings.
* **Attaches matching tokenizers** (`google/mt5-base`, `google/t5-v1_1-base`, `google-t5/t5-base`).
* **Generates comprehensive Model Cards** (`README.md`) with proper tags, usage guides, and paper citations.
* **Uploads to your Hugging Face account** with progress tracking.
* **Performs a live dry-run test** directly from Hugging Face Hub to verify everything is working!


In [ ]:
# 1. Install Dependencies & Setup
!pip -q install -U huggingface_hub transformers safetensors sentencepiece torch

import os
import sys
import glob
import json
import shutil
from pathlib import Path
from google.colab import drive, userdata
from huggingface_hub import HfApi, create_repo, upload_folder
from transformers import AutoTokenizer, AutoConfig

print("✓ Dependencies installed successfully.")


In [ ]:
# 2. Mount Google Drive & Authenticate to Hugging Face
drive.mount('/content/drive')

# Hugging Face Token:
# Option A: Save your HF write token in Colab Secrets (the Key icon on the left) named 'HF_TOKEN'
# Option B: Fallback interactive login prompt
HF_TOKEN = None
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass

if not HF_TOKEN:
    from huggingface_hub import login
    print("HF_TOKEN secret not found. Opening interactive login...")
    login()
else:
    print("✓ Authenticated using Colab Secrets (HF_TOKEN)")

api = HfApi(token=HF_TOKEN)
USER = api.whoami()['name']
print(f"✓ Logged in as Hugging Face user: https://huggingface.co/{USER}")

# Settings:
HF_PRIVATE = True   # Set to False if you want the model repositories to be Public
PREFIX = "ytasl"     # Repositories will be named: {USER}/{PREFIX}-{model_name}


In [ ]:
# 3. Locate & Inspect Checkpoints in Google Drive
POSSIBLE_PATHS = [
    "/content/drive/MyDrive/YTASL - Checkpoint",
    "/content/drive/MyDrive/YTASL-Checkpoint",
    "/content/drive/MyDrive/Checkpoints/YTASL - Checkpoint",
    "/content/drive/MyDrive/ytasl - checkpoint",
]

checkpoint_root = None
for p in POSSIBLE_PATHS:
    if os.path.exists(p):
        checkpoint_root = p
        break

if not checkpoint_root:
    print("Searching Google Drive for 'YTASL - Checkpoint'...")
    matches = glob.glob("/content/drive/MyDrive/**/YTASL - Checkpoint", recursive=True)
    if not matches:
        matches = glob.glob("/content/drive/MyDrive/**/YTASL*", recursive=True)
    if matches:
        checkpoint_root = matches[0]

assert checkpoint_root and os.path.exists(checkpoint_root), (
    "Could not find 'YTASL - Checkpoint' in Google Drive. "
    "Please verify the exact folder name inside MyDrive."
)

print(f"✓ Found Checkpoints root: {checkpoint_root}")

TARGET_MODELS = {
    "mt5-base": {
        "base_model": "google/mt5-base",
        "description": "Multilingual T5-base fine-tuned for Sign Language Translation (SLT) on YouTube-ASL keypoint features.",
    },
    "t5-v1_1-base": {
        "base_model": "google/t5-v1_1-base",
        "description": "T5 v1.1 base fine-tuned for Sign Language Translation (SLT) on YouTube-ASL keypoint features.",
    },
    "t5-base": {
        "base_model": "google-t5/t5-base",
        "description": "T5-base fine-tuned for Sign Language Translation (SLT) on YouTube-ASL keypoint features.",
    }
}

found_models = {}
for m, info in TARGET_MODELS.items():
    mp = os.path.join(checkpoint_root, m)
    if os.path.exists(mp):
        files = os.listdir(mp)
        size_mb = sum(os.path.getsize(os.path.join(mp, f)) for f in files if os.path.isfile(os.path.join(mp, f))) / 1e6
        found_models[m] = {"path": mp, "files": files, "size_mb": size_mb, **info}
        print(f"  • Found {m:15s} [{size_mb:.1f} MB]: {len(files)} files -> {files[:5]}")
    else:
        print(f"  ✗ Missing {m} in {checkpoint_root}")

print(f"\nFound {len(found_models)} of {len(TARGET_MODELS)} models to package and push.")


In [ ]:
# 4. Prepare Standalone Hugging Face Model Bundles

# Robust, self-contained modeling code (combining config and model to eliminate relative import issues)
COMBINED_MODEL_CODE = '''import torch
from torch import nn
from transformers import PretrainedConfig, PreTrainedModel, AutoModelForSeq2SeqLM, AutoConfig
from transformers.generation.utils import GenerationConfig

class SignT5Config(PretrainedConfig):
    model_type = "t5-slt"

    def __init__(
        self,
        base_model_name: str = "google-t5/t5-base",
        sign_input_dim: int = 208,
        hidden_dropout_prob: float = 0.0,
        max_length: int = 128,
        num_beams: int = 5,
        temperature: float = 1.0,
        top_k: int = 50,
        top_p: float = 1.0,
        repetition_penalty: float = 1.0,
        length_penalty: float = 0.6,
        no_repeat_ngram_size: int = 0,
        do_sample: bool = False,
        early_stopping: bool = False,
        **kwargs
    ) -> None:
        super().__init__(**kwargs)
        self.base_model_name = base_model_name
        self.sign_input_dim = sign_input_dim
        self.hidden_dropout_prob = hidden_dropout_prob
        self.max_length = max_length
        self.num_beams = num_beams
        self.temperature = temperature
        self.top_k = top_k
        self.top_p = top_p
        self.repetition_penalty = repetition_penalty
        self.length_penalty = length_penalty
        self.no_repeat_ngram_size = no_repeat_ngram_size
        self.do_sample = do_sample
        self.early_stopping = early_stopping

class T5ModelForSLT(PreTrainedModel):
    config_class = SignT5Config

    def __init__(self, config: SignT5Config):
        super().__init__(config)
        base_cfg = AutoConfig.from_pretrained(config.base_model_name)
        self.model = AutoModelForSeq2SeqLM.from_config(base_cfg)
        self.custom_linear = nn.Sequential(
            nn.Linear(config.sign_input_dim, self.model.config.d_model, bias=False),
            nn.Dropout(config.hidden_dropout_prob),
            nn.GELU(),
        )

        for layer in self.custom_linear:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)

        self.model.generation_config = GenerationConfig(
            max_length=config.max_length,
            num_beams=config.num_beams,
            temperature=config.temperature,
            top_k=config.top_k,
            top_p=config.top_p,
            repetition_penalty=config.repetition_penalty,
            length_penalty=config.length_penalty,
            no_repeat_ngram_size=config.no_repeat_ngram_size,
            do_sample=config.do_sample,
            early_stopping=config.early_stopping,
            pad_token_id=getattr(self.model.config, "pad_token_id", 0),
            bos_token_id=getattr(self.model.config, "bos_token_id", None),
            eos_token_id=getattr(self.model.config, "eos_token_id", 1),
            decoder_start_token_id=getattr(self.model.config, "pad_token_id", 0),
        )
        self.post_init()

    @torch.no_grad()
    def generate(self, sign_inputs=None, attention_mask=None, inputs_embeds=None, **kwargs):
        if inputs_embeds is None:
            inputs_embeds = self.custom_linear(sign_inputs)
        return self.model.generate(
            input_ids=None,
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            **kwargs
        )

    def forward(self, sign_inputs=None, attention_mask=None, labels=None, inputs_embeds=None, **kwargs):
        if inputs_embeds is None:
            inputs_embeds = self.custom_linear(sign_inputs)
        return self.model.forward(
            input_ids=None,
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            **kwargs
        )
'''

EXPORT_ROOT = "/content/hf_export"
os.makedirs(EXPORT_ROOT, exist_ok=True)

prepared_dirs = {}

for m, info in found_models.items():
    print(f"\n--- Packaging {m} ---")
    dest = os.path.join(EXPORT_ROOT, m)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    os.makedirs(dest, exist_ok=True)

    src = info["path"]

    # 1. Search recursively for candidate weights (> 50MB)
    candidate_weights = []
    for root_dir, _, files in os.walk(src):
        for f in files:
            fl = f.lower()
            fp = os.path.join(root_dir, f)
            if any(x in fl for x in ("optimizer", "scheduler", "rng_state", "trainer_state", "arguments")):
                continue
            if fl.endswith((".safetensors", ".bin", ".pt", ".pth")):
                try:
                    sz = os.path.getsize(fp)
                    if sz > 50 * 1024 * 1024:
                        candidate_weights.append((fp, sz))
                except Exception:
                    pass

    candidate_weights.sort(key=lambda x: x[1], reverse=True)
    if not candidate_weights:
        print(f"  ⚠️ Warning: No weights file > 50MB found in {src}!")
        # Fallback: copy any non-optimizer files
        for f in os.listdir(src):
            sfp = os.path.join(src, f)
            if os.path.isfile(sfp) and not any(x in f for x in ("optimizer", "scheduler", "rng_state")):
                shutil.copy2(sfp, os.path.join(dest, f))
    else:
        weight_path, sz = candidate_weights[0]
        print(f"  • Found primary weights: {os.path.basename(weight_path)} ({sz / 1e6:.1f} MB)")
        
        # Load and convert to standard model.safetensors
        try:
            import safetensors.torch
            if weight_path.endswith(".safetensors"):
                shutil.copy2(weight_path, os.path.join(dest, "model.safetensors"))
                print("    ✓ Copied model.safetensors")
            else:
                print("    Loading PyTorch weights to export model.safetensors...")
                raw_data = torch.load(weight_path, map_location="cpu", weights_only=False)
                if isinstance(raw_data, dict):
                    if "model" in raw_data and isinstance(raw_data["model"], dict):
                        state_dict = raw_data["model"]
                    elif "state_dict" in raw_data and isinstance(raw_data["state_dict"], dict):
                        state_dict = raw_data["state_dict"]
                    else:
                        state_dict = raw_data
                else:
                    state_dict = raw_data.state_dict()

                # Clean duplicate memory pointers for safe safetensors export
                clean_state = {}
                seen_ptrs = set()
                for k, v in state_dict.items():
                    if isinstance(v, torch.Tensor):
                        ptr = v.data_ptr()
                        if ptr in seen_ptrs:
                            clean_state[k] = v.clone()
                        else:
                            seen_ptrs.add(ptr)
                            clean_state[k] = v
                    else:
                        clean_state[k] = v
                
                safetensors.torch.save_file(clean_state, os.path.join(dest, "model.safetensors"))
                print("    ✓ Exported clean model.safetensors")
                
                # Also keep pytorch_model.bin
                shutil.copy2(weight_path, os.path.join(dest, "pytorch_model.bin"))
                del raw_data, state_dict, clean_state
                import gc
                gc.collect()
        except Exception as err:
            print(f"    ⚠️ Warning during safetensors conversion ({err}), copying as pytorch_model.bin...")
            shutil.copy2(weight_path, os.path.join(dest, "pytorch_model.bin"))

    # 2. Add custom unified modeling code
    with open(os.path.join(dest, "modeling_t5.py"), "w", encoding="utf-8") as f:
        f.write(COMBINED_MODEL_CODE)
    with open(os.path.join(dest, "configuration_t5.py"), "w", encoding="utf-8") as f:
        f.write(COMBINED_MODEL_CODE)
    with open(os.path.join(dest, "__init__.py"), "w", encoding="utf-8") as f:
        f.write("from .modeling_t5 import SignT5Config, T5ModelForSLT\n")

    # 3. Add Tokenizer from base model
    print(f"Fetching tokenizer for {info['base_model']}...")
    tokenizer = AutoTokenizer.from_pretrained(info["base_model"])
    tokenizer.save_pretrained(dest)

    # 4. Update config.json auto_map to point config and model to modeling_t5
    cfg_file = os.path.join(dest, "config.json")
    cfg = {}
    if os.path.exists(cfg_file):
        try:
            cfg = json.load(open(cfg_file, "r", encoding="utf-8"))
        except Exception:
            pass

    cfg["model_type"] = "t5-slt"
    cfg["base_model_name"] = info["base_model"]
    cfg["sign_input_dim"] = 208
    cfg["auto_map"] = {
        "AutoConfig": "modeling_t5.SignT5Config",
        "AutoModel": "modeling_t5.T5ModelForSLT",
        "AutoModelForSeq2SeqLM": "modeling_t5.T5ModelForSLT"
    }
    with open(cfg_file, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2)

    # 5. Generate Model Card README.md
    repo_id = f"{USER}/{PREFIX}-{m}"
    readme = f'''---
language:
- en
license: cc-by-4.0
tags:
- sign-language
- sign-language-translation
- slt
- t5
- youtube-asl
- pose
- mediapipe
pipeline_tag: feature-extraction
---

# {repo_id}

{info["description"]}

Trained as part of the **YouTube-ASL (YTASL)** and **T5 for SLT** research (*Železný et al.*). It maps continuous MediaPipe keypoint feature vectors (dimension 208) to written text using an adapted `{info["base_model"]}` backbone.

## Model Details
* **Base Architecture**: `{info["base_model"]}`
* **Input**: Sign language visual features `[Batch, Frames, 208]`
* **Projector**: Linear projection layer mapping 208 dims $\\to$ `{info["base_model"]}` embedding dimension with GELU activation.
* **Task**: Pose-based Sign Language Translation (SLT)

## Quickstart Usage

```python
import torch
from transformers import AutoModel, AutoTokenizer

repo_id = "{repo_id}"

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(repo_id)
model = AutoModel.from_pretrained(repo_id, trust_remote_code=True)
model.eval()

# Dummy input: batch of 1 clip, 60 frames, 208 keypoints
dummy_poses = torch.randn(1, 60, 208)

# Generate translation tokens
output_ids = model.generate(sign_inputs=dummy_poses, max_length=64)
translation = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("Predicted translation:", translation)
```

## Citation

```bibtex
@misc{{zelezny2025exploring,
  title         = {{Exploring Pose-based Sign Language Translation: Ablation Studies and Attention Insights}},
  author        = {{Zelezny, Tomas and Straka, Jakub and Javorek, Vaclav and Valach, Ondrej and Hruz, Marek and Gruber, Ivan}},
  year          = {{2025}},
  eprint        = {{2507.01532}},
  archivePrefix = {{arXiv}},
  primaryClass  = {{cs.CV}}
}}
```
'''
    with open(os.path.join(dest, "README.md"), "w", encoding="utf-8") as f:
        f.write(readme)

    prepared_dirs[m] = dest
    print(f"✓ Package ready for {m}: {len(os.listdir(dest))} files")


In [ ]:
# 5. Push All Checkpoints to Hugging Face Hub

uploaded_repos = []

for m, folder in prepared_dirs.items():
    repo_name = f"{PREFIX}-{m}"
    repo_id = f"{USER}/{repo_name}"
    print(f"\n=======================================================")
    print(f"Pushing {m} -> https://huggingface.co/{repo_id}")
    print(f"=======================================================")

    # Create model repository
    create_repo(
        repo_id=repo_id,
        repo_type="model",
        private=HF_PRIVATE,
        exist_ok=True,
        token=HF_TOKEN
    )

    # Upload folder
    upload_folder(
        folder_path=folder,
        repo_id=repo_id,
        repo_type="model",
        commit_message=f"Upload {m} checkpoint and Hugging Face model integration",
        token=HF_TOKEN,
        ignore_patterns=["*.tmp*", "*.pt.bak", ".git*"]
    )
    print(f"✓ Successfully uploaded {m} to https://huggingface.co/{repo_id}")
    uploaded_repos.append(repo_id)

print("\n" + "="*50)
print("🎉 ALL MODELS SUCCESSFULLY UPLOADED!")
for r in uploaded_repos:
    print(f" • https://huggingface.co/{r}")
print("="*50)


In [ ]:
# 6. Test & Verify Models Directly from Hugging Face Hub

import torch
import sys
import shutil
from pathlib import Path
from transformers import AutoModel, AutoTokenizer

# Clean local python sys.modules to prevent conflict with local files in current runtime
for module_name in list(sys.modules.keys()):
    if "modeling_t5" in module_name or "configuration_t5" in module_name:
        del sys.modules[module_name]

# Clean cached hub modules downloaded by transformers to force a fresh pull of unified code
hf_modules_cache = Path("~/.cache/huggingface/modules/transformers_modules").expanduser()
if hf_modules_cache.exists():
    shutil.rmtree(hf_modules_cache)
    print("✓ Cleared Hugging Face local module cache to force fresh pull.")

for repo_id in uploaded_repos:
    print(f"\nTesting load: {repo_id}...")
    try:
        # trust_remote_code=True is required to run the custom T5ModelForSLT
        tok = AutoTokenizer.from_pretrained(repo_id, token=HF_TOKEN)
        mod = AutoModel.from_pretrained(repo_id, trust_remote_code=True, token=HF_TOKEN)
        mod.eval()

        # Test forward pass with dummy pose input
        dummy_pose = torch.randn(1, 30, 208)
        with torch.no_grad():
            out_ids = mod.generate(sign_inputs=dummy_pose, max_length=16)
        text = tok.decode(out_ids[0], skip_special_tokens=True)
        print(f"  ✓ Model loaded & executed successfully! (Sample decoded text: '{text}')")
    except Exception as e:
        print(f"  ✗ Verification warning for {repo_id}: {e}")
